# ¿Dónde mira cada cabeza en CelebA con penalización de ENTROPÍA? (2026-11-02)

> **Este notebook es un duplicado de `celeba_regions.ipynb`** (el original no se toca). Es idéntico salvo que añade un cuarto modelo: `vit-5-entropy_l001-s0-CelebA` (penalización de entropía λ=0.01, sin coseno entre cabezas, **una sola corrida, semilla 0**). Con una semilla no se pueden sacar conclusiones de variabilidad: compáralo con la semilla 0 de las otras variantes y con el rango entre semillas de las demás.

---


**Pregunta.** La penalización de coseno entre cabezas, ¿hace que cada cabeza mire una **parte distinta de la cara** (ojos, nariz, boca, pelo/fondo)? En AwA2 no se pudo responder, porque sus atributos son *por clase* y no dicen *dónde* está nada. CelebA sí tiene posiciones: trae 5 puntos de referencia (*landmarks*) por cara: ojo izquierdo, ojo derecho, nariz, comisura izquierda y derecha de la boca.

**Cómo está organizado este notebook (léelo en orden):**
1. Datos: de dónde salen las imágenes de test y cómo se pasan las posiciones de los landmarks a la imagen de 64×64 que ve el modelo.
2. Regiones: cómo defino "ojos", "nariz", "boca", "cara" y "fuera de la cara" sobre la rejilla de parches 16×16.
3. Atención: cómo se extrae la atención del token [CLS] de cada cabeza en el último bloque.
4. Métricas: *enriquecimiento* por región y *parches efectivos*.
5. Resultados: tablas y figuras, comparando `unconstrained`, `constrained` y `constrained_rank` (3 semillas cada uno) con `entropy_l001` (1 semilla).
6. Vínculo con atributos (gafas, sombrero, boca abierta) y conclusión.

**Modelos usados.** Los 9 checkpoints de antes (`vit-5-{unconstrained, constrained, constrained_rank}-s{0,1,2}-CelebA`) **más** `vit-5-entropy_l001-s0-CelebA` (5 épocas, ViT de 4 bloques, 4 cabezas, imagen 64×64, parche 4×4, es decir una rejilla de 16×16 = 256 parches). El de entropía se entrena con `python run_training_notebook.py --name entropy --lambda-entropy 0.01 --seed 0` (desde `experimentation/`).

> Este notebook solo hace *inferencia* (no entrena), así que corre igual en MPS o en CPU.

## Glosario mínimo

- **Parche**: un cuadrado de 4×4 píxeles. La imagen de 64×64 se parte en 16×16 = 256 parches.
- **[CLS]**: un token extra que el ViT usa para resumir la imagen. La clasificación sale de él. Su fila de atención nos dice **a qué parches mira para decidir**.
- **Mapa de atención de una cabeza**: 256 números ≥ 0 que suman 1 (cuánto mira el [CLS] a cada parche). Se puede dibujar como una imagen 16×16.
- **Masa de atención en una región**: la suma de esos números sobre los parches de la región. Va de 0 a 1.
- **Enriquecimiento**: masa en la región ÷ fracción del área que ocupa la región. Si la cabeza mirara todo por igual, el enriquecimiento sería **1** en cualquier región. 5 significa "mira esa región 5 veces más de lo que le tocaría por tamaño". Es la métrica principal de este notebook.
- **Parches efectivos**: exp(entropía) del mapa de atención. Es un "cuántos parches usa realmente la cabeza": 256 = mira todo parejo; 1 = mira un solo parche.

## 0. Configuración

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# El notebook vive en notebooks/<carpeta>/, así que la raíz del proyecto está dos niveles arriba.
project_root = Path.cwd().resolve().parents[1]
sys.path.append(str(project_root))

from modules.figures import save_fig            # guarda la figura en PNG con un nombre ordenado
from modules.model import ViTForClassfication   # el ViT propio del proyecto

# En tu Mac se usa MPS; en cualquier otra máquina, CPU. Aquí solo hacemos inferencia, el resultado es el mismo.
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

EXPERIMENTS_DIR = project_root / "experimentation" / "experiments"
OUTPUT_DIR = EXPERIMENTS_DIR / "02112026_celeba_regions_entropy"     # aquí se guardan CSV y figuras de este notebook
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VARIANTS = ["unconstrained", "constrained", "constrained_rank", "entropy_l001"]   # sin penalizar / coseno / coseno + rango / entropía λ=0.01
SEEDS = [0, 1, 2]
SEEDS_DE = {"entropy_l001": [0]}   # el modelo de entropía solo tiene 1 corrida (semilla 0)
BLOCK = 4          # bloque (1-indexado) cuya atención analizamos: el último
N_HEADS = 4
GRID = 16          # rejilla de parches 16×16 (imagen 64 px / parche 4 px)
IMG = 64
RAW_W, RAW_H = 178, 218      # tamaño de las imágenes alineadas de CelebA
CACHE = project_root / "data" / "celeba" / "celeba_test64.npz"   # imágenes de test ya recortadas + landmarks
print("device:", device, "| raíz:", project_root)

## 1. Datos

### Por qué NO uso `list_bbox_celeba.csv`
Ese archivo trae una caja por cara, pero está en las coordenadas de la **foto original sin alinear**, no en las de las imágenes alineadas (`img_align_celeba`, 178×218) con las que se entrenó. Se nota porque ~87 % de las cajas se salen de la imagen alineada (por ejemplo la caja de la imagen 000001 mide 226×313 y la imagen mide 178×218). **No se puede usar sin reproyectarla**, y no hay forma fiable de hacerlo.

En cambio `list_landmarks_align_celeba.csv` **sí** está en las coordenadas de la imagen alineada. Por eso las regiones se definen a partir de los 5 landmarks.

### Del píxel original al píxel de 64×64
El `Dataset` del proyecto hace `CenterCrop(178)` y luego `Resize(64×64)`. Sobre una imagen de 178 de ancho × 218 de alto:
- el recorte horizontal no quita nada (178 → 178) y el vertical quita 20 px arriba y 20 px abajo (218 → 178);
- después se reduce de 178 a 64, un factor 64/178.

Entonces un landmark en (x, y) cae en **( x · 64/178 ,  (y − 20) · 64/178 )**. Es lo que hace la celda siguiente. Para no tener que leer 20 000 JPG cada vez, se guarda una caché `celeba_test64.npz` con las imágenes del conjunto de **test** (partición oficial 2, 19 962 imágenes) ya recortadas y reducidas, sus 40 atributos y sus landmarks en el marco de 64×64.

In [ ]:
def build_cache(root=project_root / "data" / "celeba", out=CACHE):
    """Construye la caché de test de CelebA (solo si no existe). Misma transformación que el Dataset del proyecto."""
    from PIL import Image
    part = pd.read_csv(root / "list_eval_partition.csv", index_col="image_id")
    ids = part.index[part["partition"] == 2].tolist()                      # 2 = test
    attrs = pd.read_csv(root / "list_attr_celeba.csv", index_col="image_id").loc[ids]
    marks = pd.read_csv(root / "list_landmarks_align_celeba.csv", index_col="image_id").loc[ids]
    img_dir = root / "img_align_celeba"
    if (img_dir / "img_align_celeba").is_dir():                             # el zip de Kaggle anida una carpeta
        img_dir = img_dir / "img_align_celeba"
    imgs = []
    for i in ids:
        im = Image.open(img_dir / i).convert("RGB")
        im = im.crop((0, 20, RAW_W, 20 + RAW_W)).resize((IMG, IMG), Image.BILINEAR)   # CenterCrop(178) + Resize(64)
        imgs.append(np.asarray(im))
    lm = marks.values.astype(np.float32).reshape(-1, 5, 2)                  # 5 puntos × (x, y)
    lm[..., 0] = lm[..., 0] * IMG / RAW_W                                   # x: solo cambia la escala
    lm[..., 1] = (lm[..., 1] - 20) * IMG / RAW_W                            # y: primero quitar los 20 px recortados
    np.savez_compressed(out, x=np.stack(imgs), attr=((attrs.values + 1) // 2).astype(np.uint8), lm=lm,
                        ids=np.array(ids), attr_names=np.array(attrs.columns))

if not CACHE.exists():
    print("construyendo caché (tarda unos minutos)...")
    build_cache()

D = np.load(CACHE, allow_pickle=True)
X8 = D["x"]                       # (N, 64, 64, 3) uint8, imágenes de test
ATTR = D["attr"]                  # (N, 40) 0/1
LM = D["lm"]                      # (N, 5, 2): ojo izq, ojo der, nariz, boca izq, boca der; (x, y) en píxeles de 64×64
ATTR_NAMES = list(D["attr_names"])
N = len(X8)
print(f"{N} imágenes de test | landmarks de ejemplo (imagen 0): \n{LM[0].round(1)}")

In [ ]:
# Comprobación visual: los landmarks deben caer sobre ojos, nariz y boca. Si no, la conversión de coordenadas estaría mal.
fig, axes = plt.subplots(1, 6, figsize=(15, 2.8))
for ax, i in zip(axes, [0, 1, 2, 3, 4, 5]):
    ax.imshow(X8[i])
    ax.scatter(LM[i, :, 0], LM[i, :, 1], c=["cyan", "cyan", "yellow", "magenta", "magenta"], s=25, edgecolors="k")
    ax.axis("off"); ax.set_title(f"imagen {i}", fontsize=9)
fig.suptitle("Landmarks sobre la imagen de 64×64: ojos (cian), nariz (amarillo), comisuras de la boca (magenta)")
plt.tight_layout()
save_fig(fig, "landmarks_check", OUTPUT_DIR, tag="celeba", project_root=project_root)
plt.show()

## 2. Regiones sobre la rejilla de parches

La atención vive en una rejilla de 16×16 parches. El centro del parche (fila *i*, columna *j*) está en el píxel (4j+2, 4i+2). Una región es un conjunto de parches. Las defino **por imagen**, a partir de sus propios landmarks (cada cara está en un sitio un poco distinto):

| Región | Definición (en píxeles de 64×64) |
|---|---|
| `ojos` | parches a ≤ 5 px de cualquiera de los dos ojos |
| `nariz` | parches a ≤ 5 px de la punta de la nariz |
| `boca` | parches a ≤ 5 px del segmento que une las dos comisuras |
| `cara (caja landmarks)` | caja que contiene los 5 landmarks, ensanchada 6 px por cada lado |
| `fuera de la cara` | todo lo que **no** está en esa caja: pelo, sombrero, cuello, fondo |
| `frente/pelo (arriba de ojos)` | parches más de 7 px por encima de la línea de los ojos |
| `abajo de la boca` | parches más de 5 px por debajo de la boca (mentón, cuello) |

Las regiones pueden solaparse (por ejemplo `ojos` está dentro de `cara`); no importa porque cada región se evalúa por separado. El radio de 5 px (≈ 1 parche y cuarto) es una decisión mía; con radios de 4–6 px las conclusiones no deberían cambiar, pero **no lo he probado**.

In [ ]:
# Coordenadas del centro de cada uno de los 256 parches (en píxeles de 64×64). Orden: fila a fila, como el modelo.
centros = (np.arange(GRID) * 4 + 2).astype(np.float32)
PX, PY = np.meshgrid(centros, centros)
PX = PX.ravel()[None]                      # (1, 256): x de cada parche
PY = PY.ravel()[None]                      # (1, 256): y de cada parche


def disco(c, r):
    """Máscara (N, 256): parches cuyo centro está a distancia <= r del punto c (N, 2)."""
    return ((PX - c[:, 0:1]) ** 2 + (PY - c[:, 1:2]) ** 2) <= r ** 2


def segmento(a, b, r):
    """Máscara (N, 256): parches a distancia <= r del segmento a-b (cada uno (N, 2))."""
    ab = b - a
    t = (((PX - a[:, 0:1]) * ab[:, 0:1] + (PY - a[:, 1:2]) * ab[:, 1:2]) / (ab ** 2).sum(1, keepdims=True)).clip(0, 1)
    return ((PX - (a[:, 0:1] + t * ab[:, 0:1])) ** 2 + (PY - (a[:, 1:2] + t * ab[:, 1:2])) ** 2) <= r ** 2


ojo_i, ojo_d, nariz, boca_i, boca_d = (LM[:, k] for k in range(5))
REGIONES = {
    "ojos": disco(ojo_i, 5) | disco(ojo_d, 5),
    "nariz": disco(nariz, 5),
    "boca": segmento(boca_i, boca_d, 5),
}
# Caja alrededor de los 5 landmarks, ensanchada 6 px
x0, x1 = LM[..., 0].min(1) - 6, LM[..., 0].max(1) + 6
y0, y1 = LM[..., 1].min(1) - 6, LM[..., 1].max(1) + 6
cara = (PX >= x0[:, None]) & (PX <= x1[:, None]) & (PY >= y0[:, None]) & (PY <= y1[:, None])
REGIONES["cara (caja landmarks)"] = cara
REGIONES["fuera de la cara"] = ~cara
y_ojos = (ojo_i[:, 1] + ojo_d[:, 1]) / 2
y_boca = (boca_i[:, 1] + boca_d[:, 1]) / 2
REGIONES["frente/pelo (arriba de ojos)"] = PY < (y_ojos[:, None] - 7)
REGIONES["abajo de la boca"] = PY > (y_boca[:, None] + 5)

# Fracción del área (de los 256 parches) que ocupa cada región, por imagen. Es el denominador del enriquecimiento.
AREA = {k: m.mean(1) for k, m in REGIONES.items()}
print("Fracción media del área por región:")
print(pd.Series({k: round(float(v.mean()), 3) for k, v in AREA.items()}))

In [ ]:
# Comprobación visual de las máscaras en tres imágenes: cada panel marca los parches de una región.
colores = {"ojos": "Reds", "nariz": "Greens", "boca": "Blues", "fuera de la cara": "Greys"}
fig, axes = plt.subplots(3, 4, figsize=(10, 7.5))
for r, i in enumerate([0, 1, 2]):
    for c, (nombre, cmap) in enumerate(colores.items()):
        axes[r, c].imshow(X8[i])
        m = REGIONES[nombre][i].reshape(GRID, GRID).astype(float)
        axes[r, c].imshow(np.kron(m, np.ones((4, 4))), cmap=cmap, alpha=0.55, vmin=0, vmax=1)   # cada parche = 4×4 px
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        if r == 0: axes[r, c].set_title(nombre)
fig.suptitle("Parches que forman cada región (coloreados) sobre tres caras de test")
plt.tight_layout()
save_fig(fig, "region_masks_check", OUTPUT_DIR, tag="celeba", project_root=project_root)
plt.show()

## 3. Extraer la atención del [CLS] en el último bloque

Para cada modelo y cada imagen de test, pasamos la imagen por el ViT con `output_attentions=True`. El modelo devuelve, por bloque, un tensor `(batch, cabezas, 257, 257)`: la atención de cada token a cada token (257 = [CLS] + 256 parches). Nos quedamos con:
- **bloque 4** (el último; `att[3]` porque la lista empieza en 0),
- **fila 0** (la del [CLS]: a quién mira el [CLS]),
- **columnas 1…256** (los parches; la columna 0 es el propio [CLS]).

Resultado por modelo: un arreglo `(19 962, 4, 256)`. Luego **renormalizo** para que, sobre los 256 parches, sume 1 (así "masa en una región" es una fracción entre 0 y 1; la atención del [CLS] a sí mismo se descarta).

**Chequeo de sanidad:** también calculo la exactitud media (promedio de los 40 atributos, umbral logit>0) y la comparo con la que guardó el entrenamiento en `metrics.json`. Si coinciden, el preprocesado de la caché es el mismo que vio el modelo durante el entrenamiento.

In [ ]:
def cargar_modelo(nombre):
    carpeta = EXPERIMENTS_DIR / nombre
    config = json.loads((carpeta / "config.json").read_text())
    modelo = ViTForClassfication(config)
    modelo.load_state_dict(torch.load(carpeta / "model_final.pt", map_location="cpu"))
    return modelo.to(device).eval()


@torch.no_grad()
def atencion_cls(modelo, lote=100):
    """Devuelve (atención CLS->parches (N, 4, 256), exactitud media de los 40 atributos)."""
    atts, ok = [], []
    for i in range(0, N, lote):
        # uint8 -> float en [-1, 1], mismo Normalize((0.5,)*3, (0.5,)*3) del entrenamiento
        x = torch.tensor(X8[i:i + lote]).permute(0, 3, 1, 2).float().div(255).sub(0.5).div(0.5).to(device)
        logits, att = modelo(x, output_attentions=True)
        # .clone(): sin esto, el slice es una "vista" y mantiene vivo todo el tensor (N,4,257,257) en memoria
        atts.append(att[BLOCK - 1][:, :, 0, 1:].clone().cpu().numpy())
        y = torch.tensor(ATTR[i:i + lote]).float().to(device)
        ok.append(((logits > 0).float() == y).float().mean().item())
    a = np.concatenate(atts)
    a = a / a.sum(-1, keepdims=True)           # renormalizar sobre los 256 parches
    return a.astype(np.float32), float(np.mean(ok))


ATT = {}                                       # (variante, semilla) -> (N, 4, 256)
chequeo = []
for v in VARIANTS:
    for s in SEEDS_DE.get(v, SEEDS):
        nombre = f"vit-5-{v}-s{s}-CelebA"
        ATT[(v, s)], acc = atencion_cls(cargar_modelo(nombre))
        acc_guardada = json.loads((EXPERIMENTS_DIR / nombre / "metrics.json").read_text())["accuracy"]
        chequeo.append({"modelo": nombre, "exactitud aquí": round(acc, 4), "exactitud en metrics.json": round(acc_guardada, 4)})
pd.DataFrame(chequeo)

## 4. Métricas por cabeza

Para cada imagen *n*, cabeza *h* y región *R*:

- **masa**(n,h,R) = Σ de la atención sobre los parches de R.
- **enriquecimiento**(n,h,R) = masa(n,h,R) ÷ (fracción de los 256 parches que ocupa R en esa imagen).
  Se promedia sobre las 19 962 imágenes. Un valor de 1 es "mira parejo"; un valor alto es "mira de más esa región". **Ojo con la lectura:** el máximo posible depende del tamaño de la región. Los ojos ocupan ~4 % del área, así que el máximo es ~25 (toda la atención en ellos); la caja de la cara ocupa ~15 %, así que su máximo es ~6.6. Por eso no hay que comparar números entre regiones de distinto tamaño, sino cada región contra 1 y entre modelos.
- **parches efectivos** = exp(−Σ aᵢ log aᵢ) con aᵢ la atención al parche i. Mide qué tan repartida está.

**Qué buscar:** cabezas con alto enriquecimiento en *una* región y pocos parches efectivos = cabezas "dedicadas" a una parte de la cara.

In [ ]:
filas = []
for (v, s), a in ATT.items():
    for h in range(N_HEADS):
        fila = {"modelo": v, "semilla": s, "cabeza": h + 1}
        for nombre, m in REGIONES.items():
            masa = (a[:, h] * m).sum(1)                                   # (N,) masa de la cabeza h en la región
            fila[nombre] = float(np.mean(masa / np.clip(AREA[nombre], 1e-6, None)))   # enriquecimiento medio
        ent = -(a[:, h] * np.log(np.clip(a[:, h], 1e-8, None))).sum(1)    # entropía por imagen
        fila["parches efectivos"] = float(np.exp(ent).mean())
        filas.append(fila)
TABLA = pd.DataFrame(filas)
TABLA.to_csv(OUTPUT_DIR / "celeba_region_enrichment.csv", index=False)
pd.set_option("display.width", 220)
print("Enriquecimiento por región (1 = parejo) y parches efectivos (de 256):")
TABLA.round(2)

In [ ]:
# Mapa de calor: una fila por (modelo, semilla, cabeza), una columna por región. El color es log2(enriquecimiento):
# azul = mira menos de lo parejo, blanco = parejo (1), rojo = mira más. El número es el enriquecimiento real.
regiones_plot = ["ojos", "nariz", "boca", "cara (caja landmarks)", "fuera de la cara"]
etiquetas = [f"{r.modelo[:5]}.. s{r.semilla} h{r.cabeza}" for r in TABLA.itertuples()]
Z = np.log2(np.clip(TABLA[regiones_plot].values, 0.05, None))
fig, ax = plt.subplots(figsize=(7.5, 13.5))
im = ax.imshow(Z, cmap="RdBu_r", vmin=-4, vmax=4, aspect="auto")
ax.set_xticks(range(len(regiones_plot))); ax.set_xticklabels(regiones_plot, rotation=30, ha="right")
ax.set_yticks(range(len(etiquetas))); ax.set_yticklabels([f"{r.modelo} s{r.semilla} cabeza {r.cabeza}" for r in TABLA.itertuples()], fontsize=7)
for i in range(Z.shape[0]):
    for j in range(Z.shape[1]):
        ax.text(j, i, f"{TABLA[regiones_plot].values[i, j]:.1f}", ha="center", va="center", fontsize=6.5)
for corte in range(4, len(TABLA), 4):                      # línea entre modelos
    ax.axhline(corte - 0.5, color="k", lw=0.4)
for corte in (12, 24, 36):                                     # línea más gruesa entre variantes
    ax.axhline(corte - 0.5, color="k", lw=2)
ax.set_title("Enriquecimiento de la atención por región (log2; número = valor real)")
fig.colorbar(im, ax=ax, fraction=0.04, label="log2(enriquecimiento)")
plt.tight_layout()
save_fig(fig, "region_enrichment_heatmap", OUTPUT_DIR, tag="celeba", project_root=project_root)
plt.show()

## 5. Cómo se ve: atención media de cada cabeza

Promedio, sobre las 19 962 imágenes, del mapa de atención 16×16 de cada cabeza (semilla 0 de cada variante). Los puntos blancos son la posición media de los landmarks (ojos, nariz, boca). Una cabeza "de ojos" debería ser una mancha sobre los dos puntos de arriba.

In [ ]:
lm_medio = LM.mean(0) / 4 - 0.5            # posición media de los landmarks en unidades de parche (el parche j está en x=j)
fig, axes = plt.subplots(4, N_HEADS, figsize=(10, 10))
for r, v in enumerate(VARIANTS):
    mapa = ATT[(v, 0)].mean(0).reshape(N_HEADS, GRID, GRID)
    for h in range(N_HEADS):
        ax = axes[r, h]
        ax.imshow(mapa[h], cmap="magma")
        ax.scatter(lm_medio[:, 0], lm_medio[:, 1], c="white", s=14, edgecolors="k", linewidths=0.5)
        ax.set_xticks([]); ax.set_yticks([])
        eff = TABLA[(TABLA.modelo == v) & (TABLA.semilla == 0) & (TABLA.cabeza == h + 1)]["parches efectivos"].item()
        ax.set_title(f"{v}  cabeza {h + 1}\n{eff:.0f} parches efectivos", fontsize=8)
fig.suptitle("Atención media del [CLS] por cabeza, bloque 4, semilla 0 (blanco = landmarks medios)")
plt.tight_layout()
save_fig(fig, "mean_attention_maps", OUTPUT_DIR, tag="celeba_seed0", project_root=project_root)
plt.show()

## 6. ¿Cuántas cabezas se "dedican" a una región? (cobertura)

Para resumir con un número por modelo, defino una cabeza como **dedicada** a una región si cumple **las dos** condiciones:
1. más del 50 % de su masa de atención (en promedio) cae dentro de esa región (entre `ojos`, `nariz`, `boca`, `fuera de la cara`);
2. usa menos de 12 parches efectivos (está concentrada, no simplemente "la región grande tiene mucha masa").

Reporto, por modelo: cuántas cabezas están dedicadas y **cuántas regiones distintas** cubren entre las 4 cabezas. Los umbrales (50 %, 12) son decisión mía; cambiarlos mueve los números pero el contraste entre variantes debería mantenerse (no lo he barrido).

In [ ]:
REG_COB = ["ojos", "nariz", "boca", "fuera de la cara"]
filas = []
for (v, s), a in ATT.items():
    dedicadas, cubre = 0, set()
    for h in range(N_HEADS):
        masa = {k: (a[:, h] * REGIONES[k]).sum(1).mean() for k in REG_COB}      # masa media por región
        top = max(masa, key=masa.get)
        ent = -(a[:, h] * np.log(np.clip(a[:, h], 1e-8, None))).sum(1)
        if masa[top] > 0.5 and np.exp(ent).mean() < 12:
            dedicadas += 1
            cubre.add(top)
    filas.append({"modelo": v, "semilla": s, "cabezas dedicadas (de 4)": dedicadas,
                  "regiones distintas cubiertas": len(cubre), "cubre": ", ".join(sorted(cubre)) or "-"})
COB = pd.DataFrame(filas)
COB.to_csv(OUTPUT_DIR / "celeba_coverage.csv", index=False)
display(COB)
display(COB.groupby("modelo")[["cabezas dedicadas (de 4)", "regiones distintas cubiertas"]].mean().loc[VARIANTS].round(2))

## 7. ¿Se relaciona con atributos? (exploratorio)

Una pregunta natural: si una cabeza mira la boca, ¿cambia cuánto mira la boca cuando la persona tiene la boca abierta? Para tres atributos con posición clara calculo, por cabeza, la **diferencia de masa en la región** entre imágenes con el atributo y sin él, y reporto la cabeza con mayor diferencia absoluta:

| Atributo | Región |
|---|---|
| `Eyeglasses` | `ojos` |
| `Wearing_Hat` | `frente/pelo (arriba de ojos)` |
| `Mouth_Slightly_Open` | `boca` |

**Esto es descriptivo**: no hay prueba de significancia, y si el atributo cambia la posición de la cara en la imagen, la atención se mueve por eso y no porque la cabeza "lea" el atributo.

In [ ]:
PARES = [("Eyeglasses", "ojos"), ("Wearing_Hat", "frente/pelo (arriba de ojos)"), ("Mouth_Slightly_Open", "boca")]
filas = []
for (v, s), a in ATT.items():
    fila = {"modelo": v, "semilla": s}
    for atributo, region in PARES:
        masa = (a * REGIONES[region][:, None, :]).sum(-1)              # (N, 4) masa de cada cabeza en la región
        con = ATTR[:, ATTR_NAMES.index(atributo)] == 1
        d = masa[con].mean(0) - masa[~con].mean(0)                     # diferencia de masa (con − sin atributo) por cabeza
        fila[f"{atributo}: |Δmasa| máx"] = float(np.abs(d).max())
        fila[f"{atributo}: cabeza"] = int(np.abs(d).argmax() + 1)
    filas.append(fila)
VINC = pd.DataFrame(filas)
VINC.to_csv(OUTPUT_DIR / "celeba_attribute_link.csv", index=False)
display(VINC.round(3))
display(VINC.groupby("modelo")[[c for c in VINC.columns if "Δmasa" in c]].mean().loc[VARIANTS].round(3))

## 8. Conclusión

(se completa tras ejecutar)